# 03 画像からPixal3Dで3D化して色を補正する

ColabのL4 GPUを選びます。入力はPNGまたはJPEGの画像1枚です。
背景があればBiRefNet-liteで切り抜き、Pixal3Dで静的な3Dモデルを生成します。
保存されたカメラ情報を使って元画像の色を投影し、画像で裏づけられない明るい線を補正します。
最終出力の `model.refined.glb` を04でリグ付けできます。
GitHubのソース一式を事前にGoogle Driveへ配置してください。

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/GameCreation/MotionStudio/10_ImageToCharacter' #@param {type:'string'}
ROOT = Path(PROJECT_DIR).expanduser().resolve()
if not (ROOT / 'tools/pixal3d_generate.py').is_file():
    raise FileNotFoundError('最新版のソース一式があるDriveフォルダーを指定してください。')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'tools'))
from colab_runtime import read_input, run_logged

## 画像と生成条件を指定する

Drive上のパスを指定します。空欄ならファイル選択画面を開きます。
初回はCUDA版のビルドと約8GBの重みの取得があり、その後にGPU生成とCPU補正が続きます。
同じランタイム内では確認済みのビルドと重みを再利用します。

In [ ]:
from uuid import uuid4

IMAGE_PATH = '' #@param {type:'string'}
SEED = 42 #@param {type:'integer'}
FOV_DEGREES = 20.0 #@param {type:'number'}

ENGINE = Path('/content/image-to-playable-pixal3d')
JOB = ENGINE / 'runs' / uuid4().hex[:12]
JOB.mkdir(parents=True)
_, payload = read_input(IMAGE_PATH)
SOURCE = JOB / 'source-image'
SOURCE.write_bytes(payload)

In [ ]:
import shutil
import subprocess


uv_path = shutil.which('uv')
if uv_path is None:
    bootstrap = ENGINE / 'uv-bootstrap'
    run_logged([sys.executable, '-m', 'pip', 'install', '--target', bootstrap,
                'uv==0.9.5'], JOB / 'uv-bootstrap.log')
    uv_path = str(bootstrap / 'bin/uv')

VENV = ENGINE / 'venv-uv'
PYTHON = VENV / 'bin/python'
if not PYTHON.is_file():
    run_logged([uv_path, 'venv', '--python', sys.executable, VENV],
               JOB / 'venv.log')

run_logged([uv_path, 'pip', 'install', '--python', PYTHON,
            'numpy==2.2.6', 'scipy==1.16.2', 'pillow==11.3.0',
            'opencv-python-headless==4.12.0.88', 'onnxruntime==1.30.0'],
           JOB / 'dependencies.log')
run_logged([PYTHON, '-c', 'import numpy, scipy, PIL, cv2, onnxruntime'],
           JOB / 'imports.log')

In [ ]:
GENERATED = JOB / 'generated'
run_logged([PYTHON, ROOT / 'tools/pixal3d_generate.py', '--base', ENGINE,
            '--image', SOURCE, '--out', GENERATED, '--seed', SEED,
            '--fov', FOV_DEGREES], JOB / 'generation.log')

In [ ]:
import os

RESULT = JOB / 'refined'
subprocess.run([str(PYTHON), '-m', 'pixal3d_refine',
                '--model', str(GENERATED / 'model.glb'),
                '--views', str(GENERATED / 'model.svviews'),
                '--output', str(RESULT)], check=True, env={
    **os.environ, 'PYTHONPATH': str(ROOT / 'tools'), 'PYTHONDONTWRITEBYTECODE': '1',
    'OPENBLAS_NUM_THREADS': '2', 'OMP_NUM_THREADS': '2',
})

In [ ]:
from IPython.display import Image, display

for name in ('00_projection.jpg', '01_colour.jpg'):
    display(Image(filename=str(RESULT / name)))

In [ ]:
import shutil

destination = ROOT / 'outputs' / ('pixal3d-' + JOB.name)
shutil.copytree(RESULT, destination)
for name in ('model.glb', 'input.png', 'generation.json'):
    shutil.copy2(GENERATED / name, destination / name)
shutil.copytree(GENERATED / 'model.svviews', destination / 'model.svviews')
for path in JOB.glob('*.log'):
    shutil.copy2(path, destination / path.name)
print('04に渡すファイル:', destination / 'model.refined.glb')